In [4]:
import pandas as pd
from copy import deepcopy
import re

## Define Data Processing Functions

In [5]:
def rename_columns(df):
    # Find columns that match the patterns
    # Rename columns that match the pattern eval_metrics/MAR[0.5][*] to eval_metrics/MAR[0.5]
    # and eval_metrics/MBR[0.5][*] to eval_metrics/MBR[0.5]
    mar_pattern = r'^eval_metrics/MAR\[0\.5\]\[\d+\]$'
    mbr_pattern = r'^eval_metrics/MBR\[0\.5\]\[\d+\]$'
    columns_to_rename = {}

    for col in df.columns:
        if re.match(mar_pattern, col):
            new_name = 'eval_metrics/MAR[0.5]'
            columns_to_rename[col] = new_name
            print(f"Renaming: {col} -> {new_name}")
        elif re.match(mbr_pattern, col):
            new_name = 'eval_metrics/MBR[0.5]'
            columns_to_rename[col] = new_name
            print(f"Renaming: {col} -> {new_name}")

    # Rename the columns
    df = df.rename(columns=columns_to_rename)
    return df

def data_processing(df):
    df['dataset'] = df['dataset'].apply(lambda x: x.lower().replace("kdd_cup_2018_with_missing", "kdd_cup_2018"))
    df['freq'] = df['dataset'].apply(lambda x: x.split('/')[1])
    df['term'] = df['dataset'].apply(lambda x: x.split('/')[2])
    df['dataset_name'] = df['dataset'].apply(lambda x: x.split('/')[0])
    df = rename_columns(df)
    return df

## Load Experiment Results

In [6]:
# Define file paths for all experiment results
file_chronos_multi = "../../results_shared/pipeline_result/gift_eval_all_terms_chronos2_multi_20260127-151017/Chronos2-GiftEval/combined_results.csv"
file_chronos_uni = "../../results_shared/pipeline_result/gift_eval_all_terms_chronos2_uni_20260127-151044/Chronos2-GiftEval/combined_results.csv"
file_moirai_multi = "../../results_shared/pipeline_result/gift_eval_all_terms_moirai_4096_multi_20260210-124604/00Moirai_multi/combined_results.csv"
file_moirai_uni = "../../results_shared/pipeline_result/gift_eval_all_terms_moirai_4096_uni_20260210-124526/00Moirai_uni/combined_results.csv"
file_toto_multi = "../../results_shared/pipeline_result/gift_eval_all_terms_toto_4096_multi_20260125-191622/ToTo-GiftEval-sample/combined_results.csv"
file_toto_uni = "../../results_shared/pipeline_result/gift_eval_all_terms_toto_4096_uni_20260125-191646/ToTo-GiftEval-sample/combined_results.csv"

# Load all CSV files
print("Loading experiment results...")
df_chronos_multi = pd.read_csv(file_chronos_multi)
df_chronos_uni = pd.read_csv(file_chronos_uni)
df_moirai_multi = pd.read_csv(file_moirai_multi)
df_moirai_uni = pd.read_csv(file_moirai_uni)
df_toto_multi = pd.read_csv(file_toto_multi)
df_toto_uni = pd.read_csv(file_toto_uni)

print("All files loaded successfully!")

Loading experiment results...
All files loaded successfully!


## Process Each DataFrame

In [7]:
# Process each dataframe
print("Processing data...\n")
df_chronos_multi = data_processing(df_chronos_multi)
df_chronos_uni = data_processing(df_chronos_uni)
df_moirai_multi = data_processing(df_moirai_multi)
df_moirai_uni = data_processing(df_moirai_uni)
df_toto_multi = data_processing(df_toto_multi)
df_toto_uni = data_processing(df_toto_uni)

print("\nAll data processed!")

Processing data...

Renaming: eval_metrics/MAR[0.5][4000] -> eval_metrics/MAR[0.5]
Renaming: eval_metrics/MBR[0.5][4000] -> eval_metrics/MBR[0.5]
Renaming: eval_metrics/MAR[0.5][4000] -> eval_metrics/MAR[0.5]
Renaming: eval_metrics/MBR[0.5][4000] -> eval_metrics/MBR[0.5]
Renaming: eval_metrics/MAR[0.5][4096] -> eval_metrics/MAR[0.5]
Renaming: eval_metrics/MBR[0.5][4096] -> eval_metrics/MBR[0.5]
Renaming: eval_metrics/MAR[0.5][4096] -> eval_metrics/MAR[0.5]
Renaming: eval_metrics/MBR[0.5][4096] -> eval_metrics/MBR[0.5]
Renaming: eval_metrics/MAR[0.5][4096] -> eval_metrics/MAR[0.5]
Renaming: eval_metrics/MBR[0.5][4096] -> eval_metrics/MBR[0.5]
Renaming: eval_metrics/MAR[0.5][4096] -> eval_metrics/MAR[0.5]
Renaming: eval_metrics/MBR[0.5][4096] -> eval_metrics/MBR[0.5]

All data processed!


## Merge All Results

In [8]:
# Merge all dataframes
df_merged = pd.concat([df_chronos_multi, df_chronos_uni, df_moirai_multi, df_moirai_uni, df_toto_multi, df_toto_uni], ignore_index=True)
df_merged['model'].replace({"00Moirai_multi": "Moirai", "00Moirai_uni": "Moirai"}, inplace=True)

print(f"Merged data shape: {df_merged.shape}")
print(f"\nColumns: {df_merged.columns.tolist()}")
print(f"\nUnique models: {df_merged['model'].unique()}")
print(f"Unique variate modes: {df_merged['variate'].unique()}")
print(f"\nDatasets per model:")
print(df_merged.groupby(['model', 'variate'])['dataset_name'].nunique())

Merged data shape: (698, 21)

Columns: ['dataset', 'variate', 'num_variates', 'model', 'eval_metrics/MAR[0.5]', 'eval_metrics/MBR[0.5]', 'eval_metrics/MSE[mean]', 'eval_metrics/MSE[0.5]', 'eval_metrics/MAE[0.5]', 'eval_metrics/MASE[0.5]', 'eval_metrics/MAPE[0.5]', 'eval_metrics/sMAPE[0.5]', 'eval_metrics/MSIS', 'eval_metrics/RMSE[mean]', 'eval_metrics/NRMSE[mean]', 'eval_metrics/ND[0.5]', 'eval_metrics/mean_weighted_sum_quantile_loss', 'dataset_slug', 'freq', 'term', 'dataset_name']

Unique models: ['Chronos2-GiftEval' 'Moirai' 'ToTo-GiftEval-sample']
Unique variate modes: ['MULTI' 'UNI']

Datasets per model:
model                 variate
Chronos2-GiftEval     MULTI      15
                      UNI        15
Moirai                MULTI      15
                      UNI        15
ToTo-GiftEval-sample  MULTI      15
                      UNI        15
Name: dataset_name, dtype: int64


In [9]:
# Preview the merged data
df_merged.head()

,dataset,variate,num_variates,model,eval_metrics/MAR[0.5],eval_metrics/MBR[0.5],eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],...,eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss,dataset_slug,freq,term,dataset_name
0,bitbrains_fast_storage/5t/short,MULTI,2,Chronos2-GiftEval,1.096216,0.684252,3.787301e+06,3.787301e+06,324.170261,0.651166,...,0.737697,13.319339,1946.098956,3.077105,0.512567,0.398537,bitbrains_fast_storage_5T__short,5t,short,bitbrains_fast_storage
1,bitbrains_fast_storage/h/short,MULTI,2,Chronos2-GiftEval,1.252003,0.743833,1.030553e+07,1.030553e+07,768.257435,0.991605,...,0.591123,19.141390,3210.222383,4.606590,1.102430,0.805784,bitbrains_fast_storage_H__short,h,short,bitbrains_fast_storage
2,bitbrains_rnd/5t/short,MULTI,2,Chronos2-GiftEval,1.266574,0.684177,3.532709e+06,3.532709e+06,243.240009,1.615298,...,0.658764,51.619002,1879.550126,3.878460,0.501927,0.398111,bitbrains_rnd_5T__short,5t,short,bitbrains_rnd
3,bitbrains_rnd/h/short,MULTI,2,Chronos2-GiftEval,1.273787,0.712225,4.003391e+06,4.003391e+06,326.047645,5.848964,...,0.560038,196.683929,2000.847571,4.416708,0.719724,0.652966,bitbrains_rnd_H__short,h,short,bitbrains_rnd
4,bizitobs_l2c/5t/short,MULTI,7,Chronos2-GiftEval,0.978810,0.366667,3.604286e-01,3.604286e-01,0.340904,0.044993,...,0.017445,0.255386,0.600357,0.027412,0.015566,0.011913,bizitobs_l2c_5T__short,5t,short,bizitobs_l2c


## Save Merged Results

In [10]:
# Create output directory if it doesn't exist
import os
output_dir = "../../results_shared/pipeline_result"
os.makedirs(output_dir, exist_ok=True)

# Save merged results
output_file = os.path.join(output_dir, "merged_experiment_results.csv")
df_merged.to_csv(output_file, index=False)

print(f"Merged results saved to: {output_file}")
print(f"Total rows: {len(df_merged)}")
print(f"Total columns: {len(df_merged.columns)}")

Merged results saved to: ../../results_shared/pipeline_result/merged_experiment_results.csv
Total rows: 698
Total columns: 21
